# Congestion Prediction Model Endpoint Test

**Model:** Capacity threshold analysis (flight density relative to area capacity)  
**Endpoints:** `GET /api/predictions/congestion`, `GET /api/predictions/bottlenecks`, `GET /api/predictions/congestion-summary`  
**Purpose:** Monitor airport area congestion and predict wait times

## Model Overview

The congestion model monitors six airport areas and classifies congestion based on flight density relative to capacity:

### Airport Areas

| Area ID | Type | Capacity | Description |
|---------|------|----------|-------------|
| `runway_28L` | runway | 2 | Primary departure/arrival runway |
| `runway_28R` | runway | 2 | Secondary runway |
| `taxiway_A` | taxiway | 5 | Main taxiway A |
| `taxiway_B` | taxiway | 5 | Main taxiway B |
| `terminal_A_apron` | apron | 10 | Terminal A gate apron |
| `terminal_B_apron` | apron | 10 | Terminal B gate apron |

### Congestion Levels

| Level | Capacity Ratio | Description |
|-------|---------------|-------------|
| `LOW` | < 50% | Normal operations |
| `MODERATE` | 50-75% | Minor delays possible |
| `HIGH` | 75-90% | Significant delays expected |
| `CRITICAL` | > 90% | Operations at capacity |

### Wait Time Estimation

| Area Type | LOW | MODERATE | HIGH | CRITICAL |
|-----------|-----|----------|------|----------|
| Runway | 0 min | 3 min | 8 min | 15 min |
| Taxiway | 0 min | 2 min | 5 min | 10 min |
| Apron | 0 min | 1 min | 3 min | 5 min |

### Flight Counting Rules

| Area Type | Counts if... |
|-----------|-------------|
| Runway | On ground or altitude < 100m |
| Taxiway | On ground and velocity > 2 m/s (moving) |
| Apron | On ground and velocity <= 5 m/s (slow/parked) |

In [ ]:
# Setup & Authentication
import json
import time

import httpx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

# --- Configuration ---
APP_BASE_URL = "https://airport-digital-twin-dev-7474645572615955.aws.databricksapps.com"

# --- Authentication ---
def get_token():
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        return w.config.authenticate()["Authorization"].removeprefix("Bearer ")
    except Exception:
        pass
    try:
        import subprocess
        result = subprocess.run(
            ["databricks", "auth", "token", "--profile", "FEVM_SERVERLESS_STABLE"],
            capture_output=True, text=True, timeout=15,
        )
        if result.returncode == 0:
            return json.loads(result.stdout)["access_token"]
    except Exception:
        pass
    raise RuntimeError("No Databricks auth token available")

TOKEN = get_token()
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print(f"Authenticated (token: {len(TOKEN)} chars)")
print(f"App: {APP_BASE_URL}")

## Endpoint Parameters

### `GET /api/predictions/congestion`

Returns congestion levels for **all** airport areas. No query parameters.

**Response schema: `CongestionListResponse`**

```json
{
  "areas": [
    {
      "area_id": "runway_28L",
      "area_type": "runway",
      "level": "moderate",
      "flight_count": 3,
      "capacity": 5,
      "wait_minutes": 3
    }
  ],
  "count": 6
}
```

### `GET /api/predictions/bottlenecks`

Returns **only HIGH and CRITICAL** congestion areas. Same response schema.

### `GET /api/predictions/congestion-summary`

Returns **both** all areas and bottlenecks in a single response.

```json
{
  "areas": [...],
  "bottlenecks": [...],
  "areas_count": 6,
  "bottlenecks_count": 1
}
```

### Response Fields

| Field | Type | Description |
|-------|------|-------------|
| `area_id` | string | Area identifier (e.g., `runway_28L`) |
| `area_type` | string | `runway`, `taxiway`, `apron`, or `terminal` |
| `level` | string | `low`, `moderate`, `high`, or `critical` |
| `flight_count` | int | Number of flights currently in this area |
| `capacity` | int | Maximum concurrent flights the area can handle |
| `wait_minutes` | int | Estimated wait time in minutes |

In [ ]:
# Call Congestion Summary Endpoint (combines congestion + bottlenecks)

url = f"{APP_BASE_URL}/api/predictions/congestion-summary"
print(f"GET {url}")

t0 = time.monotonic()
resp = httpx.get(url, headers=HEADERS, timeout=30)
elapsed_ms = int((time.monotonic() - t0) * 1000)

print(f"Status: {resp.status_code} ({elapsed_ms}ms)")

areas = []
bottlenecks = []
if resp.status_code == 200:
    data = resp.json()
    areas = data.get("areas", [])
    bottlenecks = data.get("bottlenecks", [])
    print(f"Total areas: {data.get('areas_count', len(areas))}")
    print(f"Bottlenecks (HIGH/CRITICAL): {data.get('bottlenecks_count', len(bottlenecks))}")

    # Show all areas as table
    header = "| Area | Type | Level | Flights | Capacity | Utilization | Wait |"
    sep = "|------|------|-------|---------|----------|-------------|------|"
    rows = [header, sep]
    for a in areas:
        util = (a['flight_count'] / a['capacity'] * 100) if a['capacity'] > 0 else 0
        level_icon = {"low": "  ", "moderate": "* ", "high": "**", "critical": "!!"}
        icon = level_icon.get(a['level'], '  ')
        rows.append(
            f"| {icon} {a['area_id']} | {a['area_type']} | {a['level'].upper()} "
            f"| {a['flight_count']} | {a['capacity']} | {util:.0f}% | {a['wait_minutes']} min |"
        )
    display(HTML("<pre>" + "\n".join(rows) + "</pre>"))

    if bottlenecks:
        print("\nBottlenecks requiring attention:")
        for b in bottlenecks:
            print(f"  {b['area_id']} ({b['area_type']}): {b['level'].upper()} "
                  f"- {b['flight_count']}/{b['capacity']} flights, {b['wait_minutes']} min wait")
    else:
        print("\nNo bottlenecks - all areas below HIGH threshold.")
else:
    print(f"Error: {resp.text[:500]}")

In [ ]:
# Display Results: Area Congestion Visualization

if not areas:
    print("No congestion data available. Is the app running with active flights?")
else:
    LEVEL_COLORS = {
        "low": "#2ecc71",       # green
        "moderate": "#f1c40f",  # yellow
        "high": "#e67e22",      # orange
        "critical": "#e74c3c",  # red
    }

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Utilization bar chart ---
    area_ids = [a["area_id"] for a in areas]
    utilizations = [
        (a["flight_count"] / a["capacity"] * 100) if a["capacity"] > 0 else 0
        for a in areas
    ]
    colors = [LEVEL_COLORS.get(a["level"], "gray") for a in areas]

    bars = axes[0].barh(area_ids[::-1], utilizations[::-1], color=colors[::-1], edgecolor="white")
    axes[0].set_xlabel("Utilization (%)")
    axes[0].set_xlim(0, 110)
    axes[0].set_title("Area Utilization")

    # Threshold lines
    axes[0].axvline(x=50, color="#f1c40f", linestyle="--", alpha=0.4, label="Moderate (50%)")
    axes[0].axvline(x=75, color="#e67e22", linestyle="--", alpha=0.4, label="High (75%)")
    axes[0].axvline(x=90, color="#e74c3c", linestyle="--", alpha=0.4, label="Critical (90%)")
    axes[0].legend(fontsize=8, loc="lower right")

    for bar, util in zip(bars, utilizations[::-1]):
        axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f"{util:.0f}%", va="center", fontsize=9)

    # --- Wait time bar chart ---
    wait_times = [a["wait_minutes"] for a in areas]
    bars2 = axes[1].barh(area_ids[::-1], wait_times[::-1], color=colors[::-1], edgecolor="white")
    axes[1].set_xlabel("Wait Time (minutes)")
    axes[1].set_title("Estimated Wait Times")

    for bar, wt in zip(bars2, wait_times[::-1]):
        if wt > 0:
            axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                         f"{wt} min", va="center", fontsize=9)

    # Legend
    legend_patches = [
        mpatches.Patch(color=LEVEL_COLORS[level], label=level.upper())
        for level in ["low", "moderate", "high", "critical"]
    ]
    axes[1].legend(handles=legend_patches, fontsize=8, loc="lower right")

    plt.suptitle(f"Airport Congestion ({len(areas)} areas)", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Summary
    print(f"\nSummary:")
    for level in ["low", "moderate", "high", "critical"]:
        count = sum(1 for a in areas if a["level"] == level)
        if count > 0:
            print(f"  {level.upper()}: {count} area(s)")
    total_wait = sum(a["wait_minutes"] for a in areas)
    print(f"  Total wait across all areas: {total_wait} min")